In [2]:
%%capture
!pip install torchvision

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import torchvision
from torchvision import datasets
from torch.utils.data import random_split

In [4]:
image_transformer = torchvision.models.VGG16_Weights.IMAGENET1K_V1.transforms()

In [5]:
image_data = datasets.ImageFolder("./Chameleon/test", transform = image_transformer)

In [7]:
train_set, test_set = random_split(image_data, [0.9, 0.1])

In [8]:
zero_count = 0

one_count = 0

for i in range(len(test_set)):
    
    if test_set[i][1] == 0:
        
        zero_count = zero_count + 1

    if test_set[i][1] == 1:

        one_count = one_count + 1

zero_count, one_count

(1467, 1136)

**Baseline**

In [ ]:
import torch
from torch.utils.data import DataLoader
import torchvision.models as models

device = "cuda" if torch.cuda.is_available() else "cpu"

train_loader = DataLoader(train_set, batch_size=8, shuffle=True)
test_loader = DataLoader(test_set, batch_size=8, shuffle=True)

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)  
print("Labels:", labels)


model = models.resnet18(weights=None)  
model.fc = torch.nn.Linear(model.fc.in_features, 2)  
model = model.to(device)


images = images.to(device)
with torch.no_grad():
    outputs = model(images)

print("Output shape:", outputs.shape)  

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


model.train()
total_loss = 0
correct = 0
total = 0

for images, labels in train_loader:
    images, labels = images.to(device), labels.to(device)

    optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

    total_loss += loss.item() * images.size(0)
    preds = outputs.argmax(dim=1)
    correct += (preds == labels).sum().item()
    total += images.size(0)

train_loss = total_loss / total
train_acc = correct / total

print("Train Loss:", train_loss)
print("Train Accuracy:", train_acc)



model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += images.size(0)

test_acc = correct / total
print("Test Accuracy:", test_acc)
